<a id="top"></a>
# Референсный ноутбук NUT-02: нутриентный анализ еды

Шаблон исследования для темы 2: **что сильнее влияет на качество — модель, чат или скилл
распознавания**. Ноутбук не заменяет прогоны: он показывает, как из сырых ответов получить
MAE %, Match %, t-критерий и вывод, который можно защитить.

**Как читать.** Пошаговая инструкция по этому ноутбуку:
[T2-MEMO_NOTEBOOK_GUIDE.html](T2-MEMO_NOTEBOOK_GUIDE.html) — 9 шагов с вопросами, проверками
и примерами кода.

**Данные.** В эталоне восемь блюд из задания. У команды должно быть 20+ блюд, чтобы
t-критерий был устойчивее; порог в коде считается по числу степеней свободы.


## Содержание

- [1. Вопрос и гипотеза](#sec-1)
- [2. Каталог и паспорт](#sec-2)
- [2.1. Фото и профиль блюд](#sec-2-1)
- [3. Скилл: схема и версии](#sec-3)
- [4. Прогоны: модели × чаты × скиллы](#sec-4)
- [5. MAE % и Match %](#sec-5)
- [6. t-критерий](#sec-6)
- [7. Боксплот и выбросы](#sec-7)
- [8. Схемы A–D](#sec-8)
- [9. Вывод, ограничения, сдача](#sec-9)

[↑ в начало](#top)


<a id="sec-1"></a>
## 1. Вопрос и гипотеза

**Вопрос.** Что сильнее влияет на качество нутриентного анализа — модель, чат или сам
скилл распознавания? И как доказать это числами, а не впечатлением?

**Гипотезы.**

- **H1 (скилл).** Доработка скилла v1 → v2 уменьшает MAE % сильнее, чем смена модели.
- **H2 (модель).** Разница между моделями при одном скилле больше, чем между чатами.
- **H3 (чат).** Клиент (CLI против веб) не даёт значимой разницы: инструкция одна.

**Критерий.** Сравниваем **одну переменную за раз** на одном каталоге и одном промпте.
Метрики: MAE % ≤ 15 %, Match % ≥ 80 %, t-критерий против нулевого смещения.


In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd().resolve()
while not (ROOT / "processed" / "food_catalog.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "processed" / "food_catalog.json").exists():
    raise FileNotFoundError("не найден processed/food_catalog.json — откройте ноутбук из analysis/t2")

OUT = ROOT / "out"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "violet": "#4B3F82", "lav": "#9B8ED6", "magenta": "#E62E7C",
    "coral": "#F57D64", "olive": "#6F9C4A", "sage": "#B2BE92",
    "ink": "#241E45", "muted": "#6E6690",
}
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9, "axes.titlesize": 10,
    "axes.labelsize": 9, "axes.edgecolor": "#C9BEEA",
    "axes.labelcolor": PALETTE["ink"], "xtick.color": PALETTE["muted"],
    "ytick.color": PALETTE["muted"],
})


def save_fig(fig, name: str) -> Path:
    path = FIG / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    return path


try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

print("корень:", ROOT)
print("фигуры:", FIG)


<a id="sec-2"></a>
## 2. Каталог и паспорт

Каталог — основа сравнения: одно и то же блюдо, один и тот же вес и эталон для всех
моделей и чатов. Пять блюд от каждого участника складываются в общий каталог команды,
чтобы набрать 20+ позиций для статистики.

**Паспорт.** Для каждого блюда фиксируем: фото, вес порции, калории, Б/Ж/У, состав,
источник и лицензию. Без паспорта нельзя объяснить расхождение — непонятно, чей эталон.


In [ ]:
CATALOG_PATH = ROOT / "processed" / "food_catalog.json"
catalog = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
dishes = pd.DataFrame(catalog["dishes"])

dishes["macro_kcal"] = (4 * dishes["protein"] + 9 * dishes["fat"]
                        + 4 * dishes["carbs"])
dishes["kcal_diff"] = dishes["macro_kcal"] - dishes["kcal"]

print("блюд в каталоге:", len(dishes), "| источников:",
      dishes["source"].nunique())
print("средняя разница «макросы − эталон», ккал:",
      round(dishes["kcal_diff"].abs().mean(), 1))
display(dishes[["id", "name", "weight_g", "kcal", "protein", "fat",
                "carbs", "ndi", "kcal_diff"]])


<a id="sec-2-1"></a>
## 2.1. Фото и профиль блюд

Восемь блюд эталона — не только строки таблицы. Фото помогают проверить, что модель
видит то же блюдо, а профиль КБЖУ и NDI показывает, какие позиции похожи по составу.
Фото лежат в `processed/photos/` и совпадают с `food_catalog.json` по `id`.


In [ ]:
import base64

try:
    from IPython.display import HTML, display as ipy_display
    HAS_IPY = True
except Exception:
    HAS_IPY = False

cards = []
for dish in catalog["dishes"]:
    path = ROOT / "processed" / "photos" / f"{dish['id']}.jpg"
    if path.exists():
        b64 = base64.b64encode(path.read_bytes()).decode("ascii")
        img = f'<img src="data:image/jpeg;base64,{b64}" alt="{dish["name"]}">'
    else:
        img = '<div class="np-no">нет фото</div>'
    cards.append(
        f'<figure class="np-card">{img}<figcaption><b>{dish["name"]}</b>'
        f'<span>{dish["weight_g"]} г · {dish["kcal"]} ккал · NDI {dish["ndi"]}</span>'
        '</figcaption></figure>'
    )

style = (
    '<style>'
    '.np-grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;margin:6px 0}'
    '@media(max-width:900px){.np-grid{grid-template-columns:repeat(2,minmax(0,1fr))}}'
    '.np-card{margin:0;border:1px solid #E4DDF6;border-radius:10px;overflow:hidden;background:#fff}'
    '.np-card img{width:100%;height:132px;object-fit:cover;display:block}'
    '.np-card figcaption{padding:7px 9px;display:flex;flex-direction:column;gap:2px}'
    '.np-card b{font-size:12px;color:#2E2657}'
    '.np-card span{font-size:10.5px;color:#6E6690}'
    '.np-no{height:132px;display:grid;place-items:center;background:#EDE7FB;'
    'color:#6E6690;font-size:11px}'
    '</style>'
)
html = style + '<div class="np-grid">' + "".join(cards) + "</div>"
if HAS_IPY:
    ipy_display(HTML(html))
else:
    print("фото:", ", ".join(d["name"] for d in catalog["dishes"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8))

ndi = dishes.sort_values("ndi")
colors = ["#6F9C4A" if v >= 70 else "#D79A2B" if v >= 45 else "#E62E7C"
          for v in ndi["ndi"]]
axes[0].barh(ndi["name"], ndi["ndi"], color=colors)
for i, (value, kcal) in enumerate(zip(ndi["ndi"], ndi["kcal"])):
    axes[0].text(value + 1, i, f"{value} · {kcal} ккал", va="center", fontsize=8)
axes[0].axvline(70, color="#6F9C4A", ls="--", lw=1)
axes[0].axvline(45, color="#D79A2B", ls="--", lw=1)
axes[0].set_xlim(0, 108)
axes[0].set_xlabel("NDI, 0–100")
axes[0].set_title("Нутриентная плотность: зелёный ≥ 70, янтарный 45–69")
axes[0].grid(axis="x", color="#EDE7FB")

per100 = (dishes.set_index("name")[["protein", "fat", "carbs"]]
          .div(dishes.set_index("name")["weight_g"], axis=0) * 100)
per100 = per100.sort_values("carbs")
left = np.zeros(len(per100))
for col, color, label in (("protein", "#4B3F82", "белки"),
                          ("fat", "#F57D64", "жиры"),
                          ("carbs", "#9B8ED6", "углеводы")):
    axes[1].barh(per100.index, per100[col], left=left, color=color, label=label)
    left = left + per100[col].to_numpy()
axes[1].set_xlabel("граммы на 100 г")
axes[1].set_title("Состав на 100 г: белки, жиры, углеводы")
axes[1].legend(fontsize=8, loc="lower right")
axes[1].grid(axis="x", color="#EDE7FB")

fig.tight_layout()
save_fig(fig, "nut02_dishes")
plt.show()


<a id="sec-3"></a>
## 3. Скилл: схема и версии

Скилл — главный объект задачи. Он должен возвращать **строгий JSON**, а не свободный
текст: иначе ответы нельзя сравнить между моделями. В v2 добавляем проверку порции,
аллергены, запрет выдумывать ингредиенты и отказ, если блюдо не распознано.

Схема ответа: `{components[], kcal, protein, fat, carbs, confidence, notes[]}`.
`components[]` — список объектов `{name, weight_g}`.


In [ ]:
SKILL_SCHEMA = {
    "components": list, "kcal": (int, float), "protein": (int, float),
    "fat": (int, float), "carbs": (int, float),
    "confidence": (int, float), "notes": list,
}


def validate_answer(ans) -> list[str]:
    """Проверка ответа скилла: структура, диапазоны, отказ."""
    if not isinstance(ans, dict):
        return ["ответ не объект"]
    if "error" in ans:
        return [] if ans["error"] in {"not_food", "low_confidence"} else ["неизвестный отказ"]
    errors = []
    for key, typ in SKILL_SCHEMA.items():
        if key not in ans:
            errors.append(f"нет поля {key}")
        elif not isinstance(ans[key], typ):
            errors.append(f"поле {key}: неверный тип")
    if isinstance(ans.get("components"), list):
        for comp in ans["components"]:
            if not isinstance(comp, dict) or "name" not in comp or "weight_g" not in comp:
                errors.append("компонент без name/weight_g")
    conf = ans.get("confidence")
    if isinstance(conf, (int, float)) and not 0 <= conf <= 1:
        errors.append("confidence вне 0…1")
    if isinstance(ans.get("kcal"), (int, float)) and ans["kcal"] < 0:
        errors.append("kcal < 0")
    return errors


def norm(name: str) -> str:
    return " ".join(str(name).strip().lower().replace("ё", "е").split())


def match_pct(pred_components, ref_components) -> float:
    """Доля эталонных ингредиентов, найденных в ответе (recall)."""
    pred = {norm(c["name"]) for c in pred_components if isinstance(c, dict)}
    ref = {norm(c) for c in ref_components}
    return len(pred & ref) / len(ref) if ref else float("nan")


sample_v2 = {
    "components": [{"name": "помидоры", "weight_g": 100},
                   {"name": "огурцы", "weight_g": 60},
                   {"name": "фета", "weight_g": 50},
                   {"name": "оливковое масло", "weight_g": 20}],
    "kcal": 318, "protein": 11, "fat": 25, "carbs": 12,
    "confidence": 0.82, "notes": ["порция оценена по фото"],
}
print("ошибки схемы:", validate_answer(sample_v2) or "нет")
print("match:", round(match_pct(sample_v2["components"],
      catalog["dishes"][0]["components"]) * 100, 1), "%")
print("отказ:", validate_answer({"error": "not_food", "notes": ["на фото нет блюда"]}))


<a id="sec-4"></a>
## 4. Прогоны: модели × чаты × скиллы

Один каталог, один промпт, одна переменная за раз. Минимум две модели и два чата
(CLI и Desktop/веб), ответы — в `out/runs/<вариант>/`. Если реальных прогонов ещё нет,
ноутбук собирает **демонстрационные** ответы, чтобы показать расчёт; они помечены
`runs_demo.csv` и не заменяют сдачу.


In [ ]:
RUNS_DIR = OUT / "runs"


def _read_runs():
    files = sorted(RUNS_DIR.glob("*/predictions.csv"))
    if not files:
        return None
    frames = []
    for path in files:
        frame = pd.read_csv(path)
        frame["source"] = path.parent.name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def _parse_components(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        return json.loads(value)
    except Exception:
        return []


def demo_runs() -> pd.DataFrame:
    """Синтетика для примера: сильнее всего помогает скилл v2, затем модель A."""
    rng = np.random.default_rng(SEED)
    rows = []
    for model in ("A", "B"):
        for chat in ("cli", "web"):
            for skill in ("v1", "v2"):
                for dish in catalog["dishes"]:
                    bias = (0.06 - (0.04 if model == "A" else 0)
                            - (0.05 if skill == "v2" else 0)
                            + (0.005 if chat == "web" else 0))
                    sd = (0.07 - (0.02 if skill == "v2" else 0)
                          - (0.01 if model == "A" else 0))
                    pred = dish["kcal"] * (1 + bias + rng.normal(0, sd))
                    p = (0.62 + 0.18 * (skill == "v2") + 0.10 * (model == "A")
                         + 0.02 * (chat == "web"))
                    comps = [{"name": c, "weight_g": round(dish["weight_g"] / len(dish["components"]))}
                             for c in dish["components"] if rng.random() < min(0.98, p)]
                    rows.append({
                        "model": model, "chat": chat, "skill": skill,
                        "dish_id": dish["id"], "pred_kcal": round(pred, 1),
                        "pred_components": json.dumps(comps, ensure_ascii=False),
                        "confidence": round(float(rng.uniform(0.55, 0.95)), 2),
                    })
    return pd.DataFrame(rows)


real = _read_runs()
if real is not None:
    runs = real
    RUN_SOURCE = "real"
else:
    runs = demo_runs()
    RUN_SOURCE = "demo"
    runs.to_csv(OUT / "runs_demo.csv", index=False)

runs["pred_components"] = runs["pred_components"].map(_parse_components)
runs["variant"] = runs["model"] + "-" + runs["chat"] + "-" + runs["skill"]
print("источник прогонов:", RUN_SOURCE)
print("строк:", len(runs), "| вариантов:", runs["variant"].nunique())
display(runs.groupby(["model", "chat", "skill"]).size().rename("строк").reset_index())


<a id="sec-5"></a>
## 5. MAE % и Match %

**MAE %** — средняя абсолютная ошибка калорий относительно эталона:
`mean(|pred − ref| / ref) × 100`. Порог — ≤ 15 %.

**Match %** — доля эталонных ингредиентов, найденных в ответе (recall). Порог — ≥ 80 %.
Precision отдельно не считаем, но в отчёте объясняем, если модель добавляет лишнее.


In [ ]:
ref = dishes[["id", "name", "kcal", "components"]].rename(
    columns={"id": "dish_id", "kcal": "ref_kcal", "components": "ref_components"})
data = runs.merge(ref, on="dish_id", how="left")
data["err_pct"] = (data["pred_kcal"] - data["ref_kcal"]) / data["ref_kcal"]
data["abs_err_pct"] = data["err_pct"].abs()
data["match_pct"] = [match_pct(p, r) for p, r in
                      zip(data["pred_components"], data["ref_components"])]
data.to_csv(OUT / "predictions_scored.csv", index=False)

summary = (data.groupby(["variant", "model", "chat", "skill"])
           .agg(n=("dish_id", "count"),
                mae_pct=("abs_err_pct", "mean"),
                bias_pct=("err_pct", "mean"),
                match_pct=("match_pct", "mean"))
           .reset_index())
for col in ("mae_pct", "bias_pct", "match_pct"):
    summary[col] = (summary[col] * 100).round(1)
summary = summary.sort_values("mae_pct")
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.2))
order = summary["variant"].tolist()
colors = [PALETTE["olive"] if s == "v2" else PALETTE["lav"] for s in summary["skill"]]
axes[0].barh(order, summary["mae_pct"], color=colors)
axes[0].axvline(15, color=PALETTE["magenta"], ls="--", lw=1.2)
axes[0].set_xlabel("MAE, %")
axes[0].set_title("Ошибка калорий (порог 15 %)")
axes[1].barh(order, summary["match_pct"], color=colors)
axes[1].axvline(80, color=PALETTE["magenta"], ls="--", lw=1.2)
axes[1].set_xlabel("Match, %")
axes[1].set_title("Распознанные ингредиенты (порог 80 %)")
for ax in axes:
    ax.invert_yaxis()
    ax.grid(axis="x", color="#EDE7FB")
fig.tight_layout()
save_fig(fig, "nut02_metrics")
plt.show()


### 5.1. Где ошибаются модели

Средняя ошибка скрывает блюда-провалы. Тепловая карта показывает ошибку по каждому
блюду и варианту: красное — переоценка калорий, синее — недооценка. По ней видно,
ломается ли модель на сложном составе или ошибается ровно по всем позициям.


In [ ]:
heat = data.pivot_table(index="name", columns="variant", values="err_pct") * 100
col_order = [v for v in summary["variant"].tolist() if v in heat.columns]
heat = heat.reindex(columns=col_order)

fig, ax = plt.subplots(figsize=(10.8, 4.8))
im = ax.imshow(heat.to_numpy(), cmap="RdBu_r", vmin=-35, vmax=35, aspect="auto")
ax.set_xticks(range(len(heat.columns)), heat.columns, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(len(heat.index)), heat.index, fontsize=8)
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        value = heat.iloc[i, j]
        if pd.notna(value):
            ax.text(j, i, f"{value:+.0f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(value) > 22 else "#241E45")
ax.set_title("Ошибка калорий по блюдам и вариантам, %")
fig.colorbar(im, ax=ax, fraction=.035, label="pred − ref, %")
fig.tight_layout()
save_fig(fig, "nut02_error_heatmap")
plt.show()


<a id="sec-6"></a>
## 6. t-критерий

Проверяем не «красиво ли», а есть ли **систематическое смещение**: H₀ — средняя
относительная ошибка равна нулю. Одновыборочный двусторонний критерий, порог
`|t| > t(0.975, df)`. Для 23 блюд это 2,074; в эталоне 8 блюд, поэтому порог выше —
код считает его по фактическому `df`.


In [ ]:
rows = []
for variant, group in data.groupby("variant"):
    x = group["err_pct"].dropna()
    t, p = stats.ttest_1samp(x, 0)
    crit = stats.t.ppf(0.975, len(x) - 1)
    se = x.std(ddof=1) / np.sqrt(len(x))
    rows.append({
        "variant": variant, "n": len(x), "bias_pct": x.mean() * 100,
        "ci_low_pct": (x.mean() - crit * se) * 100,
        "ci_high_pct": (x.mean() + crit * se) * 100,
        "t": t, "p": p, "crit": crit,
        "decision": "смещение значимо" if abs(t) > crit else "в пределах шума",
    })
tt = pd.DataFrame(rows).sort_values("t", key=lambda s: s.abs(), ascending=False)
display(tt.round(3))
print("вариантов со значимым смещением:", int((tt["decision"] == "смещение значимо").sum()))


### 6.1. Смещение и доверительные интервалы

Точка — среднее смещение, отрезок — 95 % CI. Если отрезок пересекает ноль,
систематического смещения нет; если нет — модель стабильно завышает или занижает калории.


In [ ]:
plot = tt.sort_values("bias_pct")
colors = ["#E62E7C" if d == "смещение значимо" else "#9B8ED6" for d in plot["decision"]]
fig, ax = plt.subplots(figsize=(9.6, 4.4))
ax.errorbar(plot["bias_pct"], range(len(plot)),
            xerr=[plot["bias_pct"] - plot["ci_low_pct"],
                  plot["ci_high_pct"] - plot["bias_pct"]],
            fmt="o", color="#4B3F82", ecolor="#C9BEEA", elinewidth=1.6, capsize=3,
            markersize=6, zorder=3)
for i, (_, row) in enumerate(plot.iterrows()):
    ax.scatter(row["bias_pct"], i, color=colors[i], s=46, zorder=4)
ax.axvline(0, color="#241E45", lw=0.9)
ax.set_yticks(range(len(plot)), plot["variant"], fontsize=8)
ax.set_xlabel("смещение калорий, % (95 % CI)")
ax.set_title("Систематическое смещение: точка и интервал")
ax.grid(axis="x", color="#EDE7FB")
fig.tight_layout()
save_fig(fig, "nut02_bias_ci")
plt.show()


<a id="sec-7"></a>
## 7. Боксплот и выбросы

Нормированное отклонение `(pred − ref) / ref` показывает не только среднее, но и разброс.
Боксплот отвечает на вопрос: модель ошибается стабильно или проваливается на отдельных
блюдах. Выбросы нужно объяснить, а не удалить: сложный состав, скрытый соус, неточный вес.


In [ ]:
order = (summary.sort_values(["skill", "model", "chat"])["variant"].tolist())
series = [data.loc[data["variant"] == v, "err_pct"].dropna() * 100 for v in order]

fig, ax = plt.subplots(figsize=(11.2, 4.8))
bp = ax.boxplot(series, positions=range(len(order)), widths=0.62,
                patch_artist=True, showfliers=True)
for patch, skill in zip(bp["boxes"], summary.sort_values(["skill", "model", "chat"])["skill"]):
    patch.set_facecolor(PALETTE["olive"] if skill == "v2" else PALETTE["lav"])
    patch.set_alpha(0.75)
for median in bp["medians"]:
    median.set_color(PALETTE["ink"])
ax.axhline(0, color=PALETTE["ink"], lw=0.8)
ax.axhline(15, color=PALETTE["magenta"], ls="--", lw=1)
ax.axhline(-15, color=PALETTE["magenta"], ls="--", lw=1)
ax.set_xticks(range(len(order)), order, rotation=25, ha="right")
ax.set_ylabel("отклонение калорий, %")
ax.set_title("Нормированное отклонение по вариантам")
ax.grid(axis="y", color="#EDE7FB")
fig.tight_layout()
save_fig(fig, "nut02_boxplot")
plt.show()

print("выбросы (|отклонение| > 25 %):")
for variant, group in data.groupby("variant"):
    bad = group.loc[group["err_pct"].abs() > 0.25, ["dish_id", "err_pct"]]
    if len(bad):
        print(" ", variant, "→", ", ".join(
            f"{r.dish_id} {r.err_pct * 100:+.0f}%" for r in bad.itertuples()))


<a id="sec-8"></a>
## 8. Схемы A–D

Схема задаёт, что именно доказывается. Сравниваем парно по блюдам, чтобы убрать
разницу между лёгкими и сложными позициями.

| Схема | Что сравниваем | Что проверяем |
|---|---|---|
| A | скилл v1 против v2 на одной модели | вклад доработки навыка |
| B | две модели на одном скилле | разница моделей |
| C | CLI против веб | влияет ли клиент |
| D | 2×2: модели × скиллы | есть ли взаимодействие |


In [ ]:
pivot = data.pivot_table(index="dish_id", columns="variant", values="err_pct")


def paired(a: str, b: str, scheme: str, question: str):
    if a not in pivot.columns or b not in pivot.columns:
        return None
    diff = (pivot[a] - pivot[b]).dropna()
    if len(diff) < 2:
        return None
    t, p = stats.ttest_rel(pivot.loc[diff.index, a], pivot.loc[diff.index, b])
    return {"scheme": scheme, "contrast": f"{a} − {b}", "question": question,
            "improvement_pp": -diff.mean() * 100, "t": t, "p": p, "n": len(diff)}


rows = [
    paired("A-cli-v2", "A-cli-v1", "A", "вклад скилла v2"),
    paired("A-cli-v2", "B-cli-v2", "B", "разница моделей A и B"),
    paired("A-web-v2", "A-cli-v2", "C", "разница чатов web и cli"),
]
# D: взаимодействие «модель × скилл» по парным разницам
d_a = (pivot["A-cli-v2"] - pivot["A-cli-v1"]).dropna()
d_b = (pivot["B-cli-v2"] - pivot["B-cli-v1"]).dropna()
common = d_a.index.intersection(d_b.index)
if len(common) >= 2:
    inter = d_a.loc[common] - d_b.loc[common]
    t, p = stats.ttest_1samp(inter, 0)
    rows.append({"scheme": "D", "contrast": "(A v2−v1) − (B v2−v1)",
                 "question": "взаимодействие модели и скилла",
                 "improvement_pp": -inter.mean() * 100, "t": t, "p": p, "n": len(common)})

factors = pd.DataFrame([r for r in rows if r])
factors["significant"] = factors["p"] < 0.05
factors.to_csv(OUT / "factors.csv", index=False)
display(factors.round(3))


### 8.1. Размер эффекта по схемам

Столбик — выигрыш в процентных пунктах MAE: больше нуля значит, что вариант слева
лучше варианта справа. Цвет показывает значимость парного критерия; маленький
столбик при p > 0,05 — это шум, а не вывод.


In [ ]:
plot = factors.sort_values("improvement_pp")
colors = ["#6F9C4A" if s else "#9B8ED6" for s in plot["significant"]]
fig, ax = plt.subplots(figsize=(9.8, 3.6))
ax.barh(plot["scheme"], plot["improvement_pp"], color=colors, height=.55)
ax.axvline(0, color="#241E45", lw=0.9)
for i, (_, row) in enumerate(plot.iterrows()):
    x = row["improvement_pp"]
    ax.text(x + (0.22 if x >= 0 else 0.12), i, f"{x:+.1f} п.п.",
            va="center", ha="left", fontsize=8,
            color="white" if x < 0 else "#241E45")
ax.margins(x=0.12)
ax.set_xlabel("выигрыш по MAE, п.п. (больше — лучше)")
ax.set_title("Схемы A–D: размер эффекта и значимость")
ax.grid(axis="x", color="#EDE7FB")
fig.tight_layout()
save_fig(fig, "nut02_schemes")
plt.show()


<a id="sec-9"></a>
## 9. Вывод, ограничения, сдача

Вывод собирается из чисел, а не из впечатления: лучший вариант, вклад скилла, вклад
модели и всё, что попало в шум. Если разница меньше ошибки, пишем «в пределах шума».


In [ ]:
best = summary.iloc[0]
factor_a = factors.loc[factors["scheme"] == "A", "improvement_pp"]
factor_b = factors.loc[factors["scheme"] == "B", "improvement_pp"]
factor_c = factors.loc[factors["scheme"] == "C", "improvement_pp"]

def pp(series) -> str:
    return f"{series.iloc[0]:+.1f} п.п." if len(series) else "н/д"

metrics = {
    "task": "NUT-02",
    "generated": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "seed": SEED,
    "runs_source": RUN_SOURCE,
    "catalog_size": int(len(dishes)),
    "variants": int(summary["variant"].nunique()),
    "best_variant": best["variant"],
    "best_mae_pct": float(best["mae_pct"]),
    "best_match_pct": float(best["match_pct"]),
    "thresholds": {"mae_pct_max": 15, "match_pct_min": 80},
    "effects_pp": {
        "skill_v2": None if not len(factor_a) else float(factor_a.iloc[0]),
        "model_a_vs_b": None if not len(factor_b) else float(factor_b.iloc[0]),
        "chat_web_vs_cli": None if not len(factor_c) else float(factor_c.iloc[0]),
    },
    "schemes": factors.to_dict(orient="records"),
}
(OUT / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2),
                                  encoding="utf-8")

print(f"Лучший вариант: {best['variant']} — MAE {best['mae_pct']} %, "
      f"Match {best['match_pct']} %")
print(f"Вклад скилла v2: {pp(factor_a)}; модели A против B: {pp(factor_b)}; "
      f"чат web против cli: {pp(factor_c)}")
print("Порог MAE ≤ 15 %:", "пройден" if best["mae_pct"] <= 15 else "не пройден")
print("Порог Match ≥ 80 %:", "пройден" if best["match_pct"] >= 80 else "не пройден")
print("Артефакты:", OUT / "metrics.json", "|", OUT / "factors.csv")


**Как читать вывод.**

- Сравниваем **модули эффектов**, а не только знак: 1–2 п.п. при разбросе 5 п.п. — шум.
- Если скилл даёт больше модели, защищайте скилл; если модель — честно скажите это.
- Чат проверяем отдельно: разница между CLI и веб не должна объясняться «удобством».

**Ограничения.** Восемь блюд в эталоне — мало для сильного вывода; состав и вес по фото
приблизительны; модели могли видеть похожие блюда в обучении. Это не медицинская
рекомендация: считаем калории и ингредиенты, а не назначаем питание.

**Что сдать.** `spec.md`, `skill/SKILL.md` с версией и CHANGELOG, `food_catalog.json`,
`PASSPORT.md`, этот ноутбук, `out/runs/`, `out/metrics.json`, `out/factors.csv`,
`report/report.html` с разделом «Традиционный способ» и границей применимости.


## Чек-лист перед сдачей

- [ ] каталог 20+ блюд, у каждого фото, вес, КБЖУ, состав и источник;
- [ ] скилл v1 и v2, строгий JSON, отказ при нераспознанном блюде;
- [ ] один каталог и один промпт, меняется одна переменная за раз;
- [ ] сырые ответы сохранены в `out/runs/<вариант>/`;
- [ ] MAE % и Match % посчитаны по каждому блюду и варианту;
- [ ] t-критерий с порогом по `df`, вывод про смещение;
- [ ] боксплот и объяснение выбросов;
- [ ] вывод: что дало больше — скилл, модель или ничего (шум);
- [ ] отчёт с «Традиционным способом» и ограничениями.
